# VadCLIP — Khuếch Đại Thích Nghi Theo Lớp (Adaptive Class-wise Rescaling)

Giai đoạn 2 trên checkpoint **baseline_ctrl 87,9** của shift-consistency, giống notebook
`train_rescale_on_shift_ctrl_kaggle.ipynb`. Khác ở **cái quyết định lớp nào được khuếch
đại và khuếch đại bao nhiêu**.

## Vấn đề của bản cũ

Bốn lớp đích — `Explosion`, `RoadAccidents`, `Shooting`, `Shoplifting` — được chọn vì
chúng có **AUC thấp nhất trên tập test**. Đó là rò rỉ tập test vào *thiết kế phương pháp*.
Và một hệ số `mu` chung chỉ nói được "có khuếch đại hay không", không nói được "lớp này
cần gấp đôi lớp kia".

## Cách làm ở đây

Trọng số tính **chỉ từ tập train**, mỗi lớp một giá trị riêng:

```
freq_c = (số video trung vị / số video lớp c) ^ p   -> chuẩn hoá theo THỨ HẠNG -> [0, 1]
diff_c = loss của θ' trên chính lớp c (trên TRAIN)  -> chuẩn hoá theo THỨ HẠNG -> [0, 1]
w_c    = clip(1 + alpha * (beta*diff_c + (1-beta)*freq_c), 1, w_max)
```

`beta` là trục ablation: **1,0** chỉ độ khó · **0,0** chỉ độ hiếm · **0,5** cả hai.
`Normal` bị ghim cứng ở 1,0 và không tham gia xếp hạng.

Ba chi tiết không phải làm cho đẹp mà là để công thức không hỏng:

| | Vì sao |
|---|---|
| Trộn bằng **phép cộng**, không phải `diff × freq` | Dạng nhân, khi bỏ `diff` đi, gán cho **mọi lớp** (kể cả `Normal`) trọng số ≥ `1 + alpha·min(freq)`. Đó là tăng learning rate nhánh A trên toàn cục, không phải ablation độ hiếm |
| Chuẩn hoá theo **thứ hạng**, không min-max | Min-max bị hai lớp cực trị quyết định hoàn toàn; một lớp outlier là đổi cả thang |
| `Normal` **ghim ở 1,0** | Khuếch đại lớp bình thường là một thí nghiệm khác, không nên xảy ra do vô tình qua công thức |

## Baseline bắt buộc: đường cong μ

Notebook này chạy `mu` = 2, 4, 6, 8, 12 **không phải để tìm mu tốt nhất**, mà để dựng
**đường cong đánh đổi**: mất bao nhiêu AUC tổng thì đổi được bao nhiêu AUC lớp đích.

Không có đường cong này thì câu "adaptive tốt hơn fixed" không phòng thủ được — người
phản biện sẽ hỏi *"cái đó có khác gì hạ mu xuống 6 không?"*. Có đường cong thì câu trả lời
là: adaptive nằm **phía trên** đường cong (được nhiều lớp đích hơn ở cùng mức mất AUC
tổng), hoặc nó không nằm phía trên và bạn biết điều đó trước khi viết bài.

## Chọn checkpoint: `--select-metric none`

Vòng trước dùng `classifier_auc`, và **đỉnh rơi vào lần chấm thứ 1/13 — tức sau 10 bước
tối ưu**, lúc model gần như vẫn là θ'. Con số đó mong manh và đổi dấu tuỳ luật chọn.

Ở đây giữ **trọng số cuối epoch**, không nhìn tập test để chọn. File `.pth` lưu ra khớp
đúng con số bạn báo cáo. Vẫn chấm 13 lần mỗi epoch để có đường cong, chỉ là không dùng nó
để chọn. Muốn đổi lại thì sửa `SELECT_METRIC` ở mục 1.

## Cần Add Input

| | |
|---|---|
| Feature | `beosngu/ucf-crime-vadclip-features` |
| **θ'** | Dataset chứa `model_baseline_ctrl.pth` (87,9) |
| CLIP *(tuỳ chọn)* | `ViT-B-16.pt`; không có thì bật Internet để tự tải |

Settings → Accelerator → **GPU**.

## 1. Cấu Hình

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK       = Path('/kaggle/working')
TEMP       = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')

# 'github' chứ KHÔNG phải 'auto'. Notebook này gần như luôn chạy với Output của một
# notebook cũ gắn làm Input (để lấy checkpoint θ'), mà Output đó có sẵn một bản sao cây
# code từ lần chạy trước. 'auto' sẽ thấy bản sao cũ đó trước và dùng luôn, không clone —
# và bạn chạy code cũ mà không hề biết. Input ở đây là để lấy checkpoint và feature,
# không phải để lấy code.
CODE_SOURCE   = 'github'     # 'auto' | 'github' | 'dataset'
# True = xoá cây code đã copy ở phiên trước rồi copy lại. Tốn vài giây, và loại hẳn khả
# năng chạy nhầm bản cũ còn sót trong /kaggle/working.
FORCE_REFRESH_CODE = True
GITHUB_REPO   = 'https://github.com/vngclinh/Finetune-VadCLIP.git'
GITHUB_BRANCH = 'main'
FEATURE_DATASET_HINT = 'beosngu/ucf-crime-vadclip-features'

CODE_OVERRIDE    = None
FEATURE_OVERRIDE = None
# Checkpoint 87,9 của shift baseline_ctrl. Ví dụ:
#   SOURCE_OVERRIDE = Path('/kaggle/input/vadclip-shift-ctrl/model_baseline_ctrl.pth')
SOURCE_OVERRIDE  = None
# Tuỳ chọn, chỉ để đối chiếu ở mục 7. KHÔNG dùng để huấn luyện.
PAPER_OVERRIDE   = None


def walk_dirs(root, maxdepth=8):
    """Duyệt thư mục theo bề rộng, CÓ đi xuyên symlink.

    Không dùng rglob: `**` của pathlib gọi is_dir(follow_symlinks=False), tức nó cố ý
    bỏ qua thư mục symlink, mà Kaggle mount dataset bằng symlink.
    """
    if not root.exists():
        return
    seen, queue = set(), [(root, 0)]
    while queue:
        directory, depth = queue.pop(0)
        try:
            key = directory.resolve()
        except OSError:
            key = directory
        if key in seen:
            continue
        seen.add(key)
        yield directory
        if depth >= maxdepth:
            continue
        try:
            queue.extend((child, depth + 1)
                         for child in sorted(directory.iterdir()) if child.is_dir())
        except (PermissionError, OSError):
            pass


def find_in_input(*markers, maxdepth=8):
    for directory in walk_dirs(INPUT_ROOT, maxdepth):
        if all((directory / m).exists() for m in markers):
            return directory
    return None


def clone_repo():
    clone_dir = WORK / 'repo'
    if (clone_dir / '.git').exists():
        print('Đã có repo, cập nhật về bản mới nhất ...')
        subprocess.run(['git', '-C', str(clone_dir), 'fetch', '--depth', '1',
                        'origin', GITHUB_BRANCH], check=True)
        subprocess.run(['git', '-C', str(clone_dir), 'reset', '--hard',
                        f'origin/{GITHUB_BRANCH}'], check=True)
    else:
        print('Clone', GITHUB_REPO, '...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH,
                        GITHUB_REPO, str(clone_dir)], check=True)
    sha = subprocess.run(['git', '-C', str(clone_dir), 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print('Commit:', sha)
    return clone_dir / 'VadCLIP'


CODE_FROM_GITHUB = False
CODE_ROOT = CODE_OVERRIDE
if CODE_ROOT is None and CODE_SOURCE != 'github':
    CODE_ROOT = find_in_input('src_rescale_ewc/ucf_train_rescale.py', 'src/model.py')
if CODE_ROOT is None and CODE_SOURCE in ('auto', 'github'):
    CODE_ROOT = clone_repo()
    CODE_FROM_GITHUB = True


def find_feature_root():
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir() and (directory / 'Vandalism').is_dir():
            return directory, 'thấy Abuse + Vandalism'
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir():
            return directory, 'chỉ thấy Abuse'
    for directory in walk_dirs(INPUT_ROOT):
        try:
            children = [d for d in directory.iterdir() if d.is_dir()]
        except (PermissionError, OSError):
            continue
        with_npy = [d for d in children if next(d.glob('*.npy'), None) is not None]
        if len(with_npy) >= 10:
            return directory, f'{len(with_npy)} thư mục con có file .npy'
    return None, None


if FEATURE_OVERRIDE is not None:
    FEATURE_ROOT, how = FEATURE_OVERRIDE, 'FEATURE_OVERRIDE đặt tay'
else:
    FEATURE_ROOT, how = find_feature_root()
print('Feature dò ra bằng:', how if FEATURE_ROOT else
      f'KHÔNG DÒ RA — Add Input dataset {FEATURE_DATASET_HINT}')

# --- Checkpoint nguồn: mọi file .pth trong /kaggle/input ---------------------------
found_pth = sorted({p for d in walk_dirs(INPUT_ROOT) for p in d.glob('*.pth')})
PAPER_MODEL = PAPER_OVERRIDE or next((p for p in found_pth if p.name == 'model_ucf.pth'), None)
candidates = [p for p in found_pth if p != PAPER_MODEL]

SOURCE_MODEL = SOURCE_OVERRIDE
if SOURCE_MODEL is None and len(candidates) == 1:
    SOURCE_MODEL = candidates[0]

print()
print('File .pth thấy trong /kaggle/input:')
for path in found_pth:
    tag = ''
    if path == PAPER_MODEL:
        tag = '   <- model_ucf.pth công bố, CHỈ để đối chiếu'
    elif path == SOURCE_MODEL:
        tag = '   <- dùng làm θ\''
    print('  ', path, tag)
if not found_pth:
    print('   (không có file .pth nào — Add Input dataset chứa checkpoint 87,9)')
elif SOURCE_MODEL is None:
    print()
    print('CÓ NHIỀU HƠN MỘT ỨNG VIÊN. Chọn tay bằng SOURCE_OVERRIDE ở đầu cell.')

# --- Code sang nơi ghi được -------------------------------------------------------
PROJECT     = WORK / 'vadclip'
SRC_DIR     = PROJECT / 'src'
RESCALE_DIR = PROJECT / 'src_rescale_ewc'
LIST_DIR    = PROJECT / 'list'
if FORCE_REFRESH_CODE and PROJECT.exists():
    print('Xoá cây code cũ:', PROJECT)
    shutil.rmtree(PROJECT)
for name, destination in (('src', SRC_DIR), ('src_rescale_ewc', RESCALE_DIR),
                          ('list', LIST_DIR)):
    if not destination.exists():
        print('Copy', name, '->', destination)
        shutil.copytree(CODE_ROOT / name, destination)
for cache in PROJECT.rglob('__pycache__'):
    shutil.rmtree(cache, ignore_errors=True)

# Kiểm NGAY tại đây, trên chính file vừa copy, và nói rõ nó đến từ đâu. Preflight ở mục
# 4 cũng kiểm, nhưng đến đó thì thông báo không còn chỉ ra được nguồn gốc nữa.
_option_text = (RESCALE_DIR / 'ucf_option_rescale.py').read_text(encoding='utf-8')
if 'adaptive_class' not in _option_text or not (RESCALE_DIR / 'adaptive_weights.py').exists():
    print('BẢN CODE CŨ: thiếu adaptive_class / adaptive_weights.py.')
    print('  Nguồn code :', CODE_ROOT)
    print('  Nếu nguồn là một thư mục trong /kaggle/input thì đó là Output của một')
    print('  notebook cũ, không phải GitHub. Đặt CODE_SOURCE = "github" rồi chạy lại.')
    raise RuntimeError('Cây code không phải bản mới.')
print('Bản code   : có adaptive_class + adaptive_weights.py')

RESULT_DIR = WORK / 'results'
LOG_DIR    = RESULT_DIR / 'logs'
MODEL_DIR  = WORK / 'models'
SCRATCH    = TEMP / 'adaptive_rescale'
for directory in (RESULT_DIR, LOG_DIR, MODEL_DIR, SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)

METRICS_CSV  = str(RESULT_DIR / 'adaptive_metrics.csv')
PERCLASS_CSV = RESULT_DIR / 'adaptive_perclass.csv'

TRAIN_LIST = str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')
TEST_LIST  = str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv')
GT_ARGS = [
    '--gt-path',         str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path',   str(LIST_DIR / 'gt_label_ucf.npy'),
]

# ====== CẤU HÌNH ======
# Giống hệt lần chạy trước, TRỪ hai thứ: select-metric và lưới chạy.
TARGET_CLASSES = ['Explosion', 'RoadAccidents', 'Shooting', 'Shoplifting']
REGULARIZER    = 'none'
LAMBDA_REG     = 0.0
LAMBDA_AUTO    = 0.0
MAX_EPOCH      = 1
LR             = '2e-6'
BATCH_SIZE     = 64
NUM_WORKERS    = 4
EVAL_STEPS     = 1280            # -> chấm mỗi 10 bước -> 13 lần trong epoch
# 'none' = giữ trọng số CUỐI epoch, không nhìn test để chọn checkpoint. Vòng trước dùng
# 'classifier_auc' và đỉnh rơi vào lần chấm 1/13, tức sau 10 bước — một con số không
# phòng thủ được. Đổi lại thành 'classifier_auc' nếu muốn lặp lại đúng giao thức cũ.
SELECT_METRIC  = 'none'
# --grad-clip, --scheduler-milestones, --scheduler-rate KHÔNG truyền: mặc định của
# ucf_option_rescale.py đã đúng là 1.0, [2], 0.1.

# --- Trọng số thích nghi ----------------------------------------------------------
ALPHA            = 6.0      # trọng số trải từ 1 đến 1+alpha
W_MAX            = 8.0      # chặn trên; ở alpha=6 nó không bao giờ chạm
FREQUENCY_POWER  = 0.5
DIFFICULTY_SOURCE = 'clasm_loss'   # loss nhánh A — đúng nhánh mà rescale tác động
BETAS            = [0.0, 0.5, 1.0]  # 0 = chỉ độ hiếm | 1 = chỉ độ khó | 0.5 = cả hai

# --- Lưới chạy --------------------------------------------------------------------
# Ba cấu hình đầu là bộ ba chính (đối chứng / adaptive / mu cao nhất), xếp trước để nếu
# phiên Kaggle hết giờ giữa chừng thì phần quan trọng nhất đã xong. Năm mức mu dựng
# đường cong đánh đổi — đó là baseline bắt buộc, không phải phần thêm cho vui.
SEEDS = [234, 1234]
CONFIGS = [
    ('ctrl',      'off',            1.0,  None),
    ('adapt_b05', 'adaptive_class', 1.0,  0.5),
    ('mu12',      'class',          12.0, None),
    ('mu6',       'class',          6.0,  None),
    ('mu2',       'class',          2.0,  None),
    ('mu4',       'class',          4.0,  None),
    ('mu8',       'class',          8.0,  None),
    ('adapt_b0',  'adaptive_class', 1.0,  0.0),
    ('adapt_b1',  'adaptive_class', 1.0,  1.0),
]
RUNS = [(f'{name}_s{seed}', mode, mu, beta, seed)
        for name, mode, mu, beta in CONFIGS for seed in SEEDS]

# Để trống = chạy tất cả. Muốn chia làm nhiều phiên thì điền tag vào đây; mỗi lần chạy
# xong một tag là file .pth của nó nằm trong Output, lần sau notebook tự bỏ qua.
ONLY_TAGS = []
# ======================================================

def beta_tag(beta):
    return f'{beta:g}'.replace('.', '')


WEIGHT_FILE = {beta: MODEL_DIR / f'w_beta{beta_tag(beta)}.json' for beta in BETAS}
STATS_FILE  = WEIGHT_FILE[BETAS[len(BETAS) // 2]]   # file được đo thật; các beta khác suy ra
DIFFICULTY_CSV = RESULT_DIR / 'adaptive_difficulty.csv'

sys.path.insert(0, str(RESCALE_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))     # utils/ nằm trong src/, preflight cần nó
os.chdir(RESCALE_DIR)
PY = [sys.executable, '-u']

print()
print('Nguồn code    :', 'GitHub (' + GITHUB_BRANCH + ')' if CODE_FROM_GITHUB
      else 'Dataset ' + str(CODE_ROOT))
print('Feature       :', FEATURE_ROOT)
print("θ' (nguồn)    :", SOURCE_MODEL)
print('Đối chiếu     :', PAPER_MODEL or '(không có model_ucf.pth, bỏ qua)')
print('Kết quả       :', RESULT_DIR)
print()
print('Trọng số      : alpha', ALPHA, '| w_max', W_MAX, '| beta', BETAS,
      '| độ khó từ', DIFFICULTY_SOURCE)
print('Đường cong μ  :', [c[2] for c in CONFIGS if c[1] == 'class'])
print('Lịch          :', MAX_EPOCH, 'epoch | lr', LR,
      '| grad_clip 1.0 và milestones [2] theo mặc định script')
print('Chấm điểm     : eval_steps', EVAL_STEPS, '-> 13 lần | giữ trọng số:', SELECT_METRIC)
print('Lớp đích      :', ', '.join(TARGET_CLASSES), '(CHỈ để gộp số khi báo cáo)')
print()
to_run = [r[0] for r in RUNS if not ONLY_TAGS or r[0] in ONLY_TAGS]
print('Sẽ chạy       :', len(to_run), 'lần —', ', '.join(to_run))
print('Ước tính      : ~15 phút/lần ->', f'{len(to_run) * 15 / 60:.1f} giờ.',
      'Hết giờ phiên thì chạy lại notebook, các tag đã xong sẽ tự bỏ qua.')

## 2. Dependencies

In [ ]:
!pip -q install ftfy regex

## 3. Nạp Sẵn Trọng Số CLIP

Không bắt buộc. Có `ViT-B-16.pt` trong Dataset thì đỡ tải 335 MB mỗi phiên.

In [ ]:
CLIP_SHA256 = '5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f'
cache_dir = Path.home() / '.cache' / 'clip'
cache_dir.mkdir(parents=True, exist_ok=True)
target = cache_dir / 'ViT-B-16.pt'

source = next((d / 'ViT-B-16.pt' for d in walk_dirs(INPUT_ROOT)
               if (d / 'ViT-B-16.pt').exists()), None)
if target.exists():
    print('Đã có sẵn trong cache:', target)
elif source is None:
    print('Không thấy ViT-B-16.pt. CLIP sẽ tự tải (cần Internet: On).')
else:
    print('Copy', source, '->', target)
    shutil.copy2(source, target)

if target.exists():
    import hashlib
    print('SHA256 khớp:', hashlib.sha256(target.read_bytes()).hexdigest() == CLIP_SHA256)

## 4. Preflight

Dừng sớm khi thiếu thứ gì, thay vì chạy 30 phút rồi mới đổ. Ngoài các mục cũ, ở đây kiểm
thêm ba thứ của nhánh adaptive: có `adaptive_weights.py`, `ucf_option_rescale.py` có
`--class-weight-file`, và `--rescale-mode` có nhận `adaptive_class`.

In [ ]:
import csv

import numpy as np
import torch

_preflight_done = False
GT_MISSING = []


def preflight(force=False):
    global _preflight_done, GT_MISSING
    if _preflight_done and not force:
        return True

    problems = []

    if not torch.cuda.is_available():
        problems.append('Không có GPU. Settings -> Accelerator -> GPU.')
    if FEATURE_ROOT is None:
        problems.append(f'Không dò ra dataset feature ({FEATURE_DATASET_HINT}).')
    if SOURCE_MODEL is None or not Path(SOURCE_MODEL).exists():
        problems.append(
            "THIẾU checkpoint nguồn θ'. Đây là giai đoạn 2, nó tinh chỉnh TỪ một mô hình "
            'đã hội tụ. Tạo Dataset chứa model_<tag>.pth (87,9) của '
            'train_shift_consistency_kaggle.ipynb rồi Add Input, hoặc đặt SOURCE_OVERRIDE.')

    need = [RESCALE_DIR / n for n in
            ['ucf_train_rescale.py', 'ucf_option_rescale.py', 'ucf_eval_perclass.py',
             'losses.py', 'dataset_rescale.py', 'evaluation.py', '_bootstrap.py',
             'adaptive_weights.py', 'ucf_train_difficulty.py',
             'tests/test_losses.py', 'tests/test_adaptive_weights.py']]
    need += [SRC_DIR / n for n in
             ['model.py', 'utils/tools.py', 'utils/layers.py',
              'utils/ucf_detectionMAP.py', 'clip/clip.py',
              'clip/bpe_simple_vocab_16e6.txt.gz']]
    need += [Path(TRAIN_LIST), Path(TEST_LIST)]
    for path in need:
        if not path.exists():
            problems.append(f'Thiếu file: {path}')

    if (RESCALE_DIR / 'ucf_option_rescale.py').exists():
        import importlib
        import ucf_option_rescale
        importlib.reload(ucf_option_rescale)
        known = {a.dest for a in ucf_option_rescale.parser._actions}
        for name in sorted({'mu', 'rescale_mode', 'target_classes', 'select_metric',
                            'grad_clip', 'use_pretrained_model', 'class_weight_file',
                            'adaptive_beta', 'adaptive_alpha', 'from_statistics'} - known):
            problems.append(f'ucf_option_rescale.py thiếu --{name.replace("_", "-")} — bản cũ.')
        mode_action = next(a for a in ucf_option_rescale.parser._actions
                           if a.dest == 'rescale_mode')
        if 'adaptive_class' not in (mode_action.choices or []):
            problems.append('--rescale-mode không nhận adaptive_class — bản cũ.')

    try:
        from utils.layers import DistanceAdj
        probe = DistanceAdj()
        if probe(2, 32).device.type != 'cpu':
            problems.append('utils/layers.py là BẢN CŨ: DistanceAdj ghi cứng .to("cuda").')
        del probe
    except Exception as error:
        problems.append(f'Không nạp được utils/layers.py: {error}')

    GT_MISSING = [LIST_DIR / n for n in ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')
                  if not (LIST_DIR / n).exists()]

    if problems:
        print('PREFLIGHT KHÔNG ĐẠT:')
        for problem in problems:
            print('  -', problem)
        raise RuntimeError('Sửa các mục trên rồi chạy lại cell này.')

    # --- Nạp thử checkpoint nguồn vào đúng kiến trúc --------------------------------
    from model import CLIPVAD
    import ucf_option_rescale
    args = ucf_option_rescale.parser.parse_args([])
    model = CLIPVAD(args.classes_num, args.embed_dim, args.visual_length,
                    args.visual_width, args.visual_head, args.visual_layers,
                    args.attn_window, args.prompt_prefix, args.prompt_postfix, 'cpu')
    state = torch.load(SOURCE_MODEL, map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    missing, unexpected = model.load_state_dict(state, strict=False)
    real_missing = [k for k in missing if not k.startswith('clipmodel.')]
    if real_missing or unexpected:
        raise RuntimeError(
            "Checkpoint nguồn KHÔNG khớp kiến trúc.\n"
            f'  thiếu   : {real_missing[:6]}\n  thừa    : {list(unexpected)[:6]}')
    del model, state

    print()
    print('PREFLIGHT ĐẠT')
    print('  GPU          :', torch.cuda.get_device_name(0))
    print('  torch        :', torch.__version__)
    print("  θ' nạp thử   : khớp kiến trúc CLIPVAD, không thiếu không thừa tham số")
    print('  layers.py    : bản đã vá')
    print('  adaptive     : có --rescale-mode adaptive_class')
    if GT_MISSING:
        print('  Thiếu ground truth (mục 4.1 sẽ sinh lại):', [p.name for p in GT_MISSING])
    else:
        gt = np.load(LIST_DIR / 'gt_ucf.npy')
        print('  gt_ucf.npy   :', len(gt), 'frame |', int(gt.sum()), 'frame bất thường')

    _preflight_done = True
    return True


preflight()

### 4.1. Sinh Lại Ground Truth (chỉ khi mục 4 báo thiếu)

In [ ]:
if GT_MISSING:
    subprocess.run([str(x) for x in PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--annotation', str(LIST_DIR / 'Temporal_Anomaly_Annotation.txt'),
        '--output-dir', LIST_DIR,
    ]], check=True)
    preflight(force=True)
else:
    print('Đã có đủ ground truth, bỏ qua.')

## 5. Unit Test Và Hàm Chạy

`test_losses.py` giữ nguyên vai trò cũ: với `mu = 1` các loss ở đây phải **trùng từng bit**
với `ucf_train.py`, nếu không thì mọi so sánh đều là so với cột mốc đã bị xê dịch.

`test_adaptive_weights.py` thêm phần công thức trọng số. Ba test trong đó là về **thí
nghiệm** chứ không phải về code: `beta = 0` phải để lớp phổ biến nhất ở đúng 1,0; một lớp
nhiều video nhưng khó phải được độ khó xếp cao và độ hiếm xếp thấp; và trọng số phải đến
đúng cột logits của nhánh A, **chính xác** chứ không chỉ "lớn hơn".

In [ ]:
import time


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_train_cmd(tag, rescale_mode, mu, beta, seed):
    """Mọi lần chạy khác nhau đúng ở phần rescale, không ở chỗ nào khác.

    Không truyền --grad-clip / --scheduler-milestones / --scheduler-rate: mặc định của
    ucf_option_rescale.py đã là 1.0 / [2] / 0.1, đúng thứ lần chạy 88,19 đã dùng.
    Cũng không truyền --use-pretrained-model: mặc định true, tức có nạp θ'.
    """
    adaptive = ['--class-weight-file', WEIGHT_FILE[beta]] if rescale_mode == 'adaptive_class' else []
    return PY + [
        'ucf_train_rescale.py',
        '--pretrained-model-path', SOURCE_MODEL,
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--target-classes', *TARGET_CLASSES,
        '--seed', seed,
        '--rescale-mode', rescale_mode,
        '--mu', mu,
        '--regularizer', REGULARIZER,
        '--lambda-reg', LAMBDA_REG,
        '--lambda-auto', LAMBDA_AUTO,
        '--max-epoch', MAX_EPOCH,
        '--lr', LR,
        '--batch-size', BATCH_SIZE,
        '--num-workers', NUM_WORKERS,
        '--pin-memory', 'true',
        '--eval-steps', EVAL_STEPS,
        '--select-metric', SELECT_METRIC,
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV,
        '--output-model-path', MODEL_DIR / f's2_{tag}.pth',
        '--checkpoint-path',      SCRATCH / f'checkpoint_{tag}.pth',
        '--save-cur-path',        SCRATCH / f'model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', SCRATCH / f'epoch_{tag}',
        *adaptive,
    ]


def train_1epoch(tag, rescale_mode, mu, beta, seed):
    preflight()
    output_path = MODEL_DIR / f's2_{tag}.pth'
    if output_path.exists():
        print(f'[bỏ qua] {output_path} đã tồn tại. Xoá file nếu muốn chạy lại.')
        return None
    started = time.time()
    output = run_command(build_train_cmd(tag, rescale_mode, mu, beta, seed),
                         log_name=f'train_{tag}.log')
    print(f'Xong sau {(time.time() - started) / 60:.1f} phút')
    return output


run_command(PY + ['tests/test_losses.py'], log_name='test_losses.log')
run_command(PY + ['tests/test_adaptive_weights.py'], log_name='test_adaptive_weights.log')

## 6. Bước 0 — Đo Độ Khó Trên Tập Train

Chạy θ' trên **toàn bộ tập train** và ghi lại, theo từng lớp, chính các loss mà trainer
tối ưu. Không đụng đến tập test.

Một lần đo là đủ cho cả ba nhánh `beta`: `beta` chỉ ảnh hưởng phép tính **sau** khi đo,
nên hai file còn lại sinh ra bằng `--from-statistics`, không cần GPU.

### Đọc bảng in ra như thế nào

Dòng quan trọng nhất không phải trọng số mà là **hệ số tương quan hạng** ở cuối:

```
Spearman(train_video_count, difficulty_raw)
```

θ' được train trên chính tập train này. Lớp ít video thì nhận ít lượt cập nhật gradient
nên giữ train loss cao — tức **"độ khó" có thể chỉ là "độ hiếm" đội lốt**. Nếu |ρ| lớn
(≳ 0,6) thì `beta = 0` và `beta = 1` không phải hai tín hiệu độc lập, và phải nói thẳng
điều đó trong bài thay vì trình bày như một ablation hai chiều. Script tự in ra kết luận.

In [ ]:
preflight()

if STATS_FILE.exists():
    print('Đã có', STATS_FILE.name, '- bỏ qua phép đo. Xoá file nếu muốn đo lại.')
else:
    run_command(PY + [
        'ucf_train_difficulty.py',
        '--pretrained-model-path', SOURCE_MODEL,
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--batch-size', BATCH_SIZE,
        '--num-workers', NUM_WORKERS,
        '--pin-memory', 'true',
        '--difficulty-source', DIFFICULTY_SOURCE,
        '--adaptive-alpha', ALPHA,
        '--adaptive-w-max', W_MAX,
        '--adaptive-frequency-power', FREQUENCY_POWER,
        '--adaptive-beta', BETAS[len(BETAS) // 2],
        '--difficulty-output', STATS_FILE,
        '--difficulty-csv', DIFFICULTY_CSV,
    ], log_name='difficulty.log')

# Các beta còn lại: chỉ là số học trên phép đo vừa rồi, không chạy lại model.
for beta in BETAS:
    path = WEIGHT_FILE[beta]
    if path.exists():
        print('Đã có', path.name)
        continue
    print('#' * 90)
    run_command(PY + [
        'ucf_train_difficulty.py',
        '--from-statistics', STATS_FILE,
        '--difficulty-source', DIFFICULTY_SOURCE,
        '--adaptive-alpha', ALPHA,
        '--adaptive-w-max', W_MAX,
        '--adaptive-frequency-power', FREQUENCY_POWER,
        '--adaptive-beta', beta,
        '--difficulty-output', path,
    ], log_name=f'weights_beta{beta_tag(beta)}.log')

### 6.1. Ba Vector Trọng Số Cạnh Nhau

Bảng này là thứ nên đưa vào bài. Nó cho thấy `beta` thực sự đổi thứ tự ưu tiên, chứ không
phải ba biến thể của cùng một danh sách.

Cột `đích?` đánh dấu bốn lớp mà bản cũ chọn tay. Nếu `beta = 0,5` xếp hạng ra gần đúng bốn
lớp đó thì adaptive ≈ fixed, và bạn biết điều đó **trước khi** chạy 18 lần — claim vẫn
viết được (bỏ được tập test khỏi thiết kế) nhưng phải hạ kỳ vọng về mức tăng.

In [ ]:
import json

import pandas as pd

table = {}
for beta in BETAS:
    table[f'beta={beta:g}'] = json.loads(WEIGHT_FILE[beta].read_text(encoding='utf-8'))['weights']

weights_frame = pd.DataFrame(table)
weights_frame['đích?'] = ['x' if c in TARGET_CLASSES else '' for c in weights_frame.index]
if DIFFICULTY_CSV.exists():
    stats = pd.read_csv(DIFFICULTY_CSV).set_index('class_name')
    for column in ('train_video_count', 'mean_CLASM_loss'):
        if column in stats.columns:
            weights_frame[column] = stats[column]
weights_frame = weights_frame.sort_values('beta=0.5', ascending=False)

print('=== TRỌNG SỐ THEO TỪNG NHÁNH BETA ===')
print(weights_frame.round(3).to_string())
weights_frame.to_csv(RESULT_DIR / 'adaptive_weight_table.csv')

targets_by_beta = {name: list(weights_frame[name].nlargest(4).index) for name in table}
print()
print('Bốn lớp nặng nhất theo từng beta (so với bốn lớp chọn tay của bản cũ):')
print('  chọn tay   :', sorted(TARGET_CLASSES))
for name, top in targets_by_beta.items():
    overlap = len(set(top) & set(TARGET_CLASSES))
    print(f'  {name:<11}:', sorted(top), f'  -> trùng {overlap}/4')
print()
print('Saved:', RESULT_DIR / 'adaptive_weight_table.csv')

## 7. Các Lần Chạy

18 lần: 9 cấu hình × 2 seed. Mỗi lần khoảng 15 phút, tổng khoảng **4,5 giờ**.

Chạy xong tag nào thì file `.pth` của tag đó nằm trong Output; chạy lại notebook sẽ tự bỏ
qua nó. Muốn chia nhỏ ra nhiều phiên thì điền `ONLY_TAGS` ở mục 1.

In [ ]:
for tag, mode, mu, beta, seed in RUNS:
    if ONLY_TAGS and tag not in ONLY_TAGS:
        continue
    label = f'mode={mode}' + (f'  mu={mu}' if mode == 'class' else '') + \
            (f'  beta={beta}' if mode == 'adaptive_class' else '')
    print('#' * 90)
    print(f"{tag}  |  {label}  seed={seed}  |  θ' = {Path(SOURCE_MODEL).name}")
    print('#' * 90)
    train_1epoch(tag, mode, mu, beta, seed)

done = [t for t, *_ in RUNS if (MODEL_DIR / f's2_{t}.pth').exists()]
print()
print(f'Đã có trọng số: {len(done)}/{len(RUNS)}')
missing = [t for t, *_ in RUNS if t not in done]
print('Còn thiếu     :', missing if missing else '(không còn)')

## 8. Chấm Điểm Theo Lớp

Một lượt duy nhất trên tập test, cho mọi checkpoint. Đây là **lần đầu và lần duy nhất** tập
test được dùng trong toàn bộ quy trình — trọng số ở mục 6 sinh ra hoàn toàn từ tập train.

In [ ]:
model_specs = [f'source={SOURCE_MODEL}']
if PAPER_MODEL and Path(PAPER_MODEL).exists():
    model_specs.append(f'paper={PAPER_MODEL}')
model_specs += [f'{tag}=' + str(MODEL_DIR / f's2_{tag}.pth') for tag, *_ in RUNS]

dropped = [s for s in model_specs if not Path(s.split('=', 1)[1]).exists()]
model_specs = [s for s in model_specs if Path(s.split('=', 1)[1]).exists()]
for spec in dropped:
    print('BỊ LOẠI (không tìm thấy file):', spec)
print('Sẽ chấm điểm', len(model_specs), 'mô hình:', [s.split('=')[0] for s in model_specs])

run_command(PY + [
    'ucf_eval_perclass.py',
    '--feature-root', FEATURE_ROOT,
    '--test-list', TEST_LIST,
    *GT_ARGS,
    '--target-classes', *TARGET_CLASSES,
    '--eval-model-paths', *model_specs,
    '--eval-output', str(PERCLASS_CSV),
], log_name='eval_perclass.log')

## 9. Kết Quả — Đường Cong Đánh Đổi

Bảng chính không phải "cái nào AUC cao nhất" mà là **cái giá và cái được**:

```
trục ngang : Δ AUC tổng    so với đối chứng cùng seed   (cái giá, thường âm)
trục dọc   : Δ AUC lớp đích so với đối chứng cùng seed  (cái được)
```

Năm mức `mu` vẽ ra một đường. Câu hỏi duy nhất đáng hỏi là: **điểm adaptive nằm trên hay
nằm trên đường đó?**

Sàn nhiễu đã đo được ở cấu hình này (1 epoch, lr 2e-6, hai lần chạy đối chứng khác seed):

| | sàn nhiễu |
|---|---|
| `classifier_auc` | **0,030** |
| `target_auc_c` | **0,049** |
| `rest_auc_c` | 0,062 |
| `target_ap_a` | 0,659 |
| `avg_mAP` | **0,151** |

Chênh lệch nhỏ hơn các mức này là nhiễu, không phải kết quả. `avg_mAP` đáng chú ý: vòng
trước đo được +0,27 với sàn 0,151 — tức là **ngay sát mức nhiễu**, nên đừng xây kết luận
chính lên nó.

In [ ]:
import numpy as np
import pandas as pd

NOISE = {'classifier_auc': 0.030, 'target_auc_c': 0.049, 'rest_auc_c': 0.062,
         'target_ap_a': 0.659, 'avg_mAP': 0.151}
COLS = ['classifier_auc', 'target_auc_c', 'target_ap_c', 'rest_auc_c',
        'avg_mAP', 'normal_fpr@0.5']

summary_path = Path(str(PERCLASS_CSV).replace('.csv', '_summary.csv'))
if not summary_path.exists():
    raise SystemExit('Chưa có bảng theo lớp. Chạy mục 8 trước.')

summary = pd.read_csv(summary_path).set_index('run')
cols = [c for c in COLS if c in summary.columns]

if 'source' in summary.index:
    print(f"θ' (checkpoint tự train) : classifier_auc "
          f"{summary.loc['source', 'classifier_auc']:.2f}")
    print()

# --- Delta so với đối chứng CÙNG SEED, rồi lấy trung bình hai seed -------------------
rows = []
for name, mode, mu, beta in CONFIGS:
    if name == 'ctrl':
        continue
    deltas = []
    for seed in SEEDS:
        run, base = f'{name}_s{seed}', f'ctrl_s{seed}'
        if run in summary.index and base in summary.index:
            deltas.append(summary.loc[run, cols] - summary.loc[base, cols])
    if not deltas:
        continue
    mean = pd.concat(deltas, axis=1).mean(axis=1)
    rows.append({'cấu hình': name, 'loại': 'adaptive' if mode == 'adaptive_class' else 'fixed μ',
                 'μ/β': beta if mode == 'adaptive_class' else mu, 'n_seed': len(deltas),
                 **{c: round(float(mean[c]), 3) for c in cols}})

if not rows:
    raise SystemExit('Chưa đủ cặp (cấu hình, đối chứng) cùng seed.')

delta = pd.DataFrame(rows).set_index('cấu hình')
print('=== Δ SO VỚI ĐỐI CHỨNG CÙNG SEED (trung bình 2 seed, trọng số CUỐI epoch) ===')
print(delta.to_string())
delta.to_csv(RESULT_DIR / 'adaptive_delta.csv')

# --- Có vượt sàn nhiễu không --------------------------------------------------------
print()
print('=== BỘI SỐ SÀN NHIỄU (|Δ| / sàn; dưới 1,0 là không phân biệt được với nhiễu) ===')
ratio = pd.DataFrame({c: (delta[c].abs() / NOISE[c]).round(1)
                      for c in cols if c in NOISE}, index=delta.index)
print(ratio.to_string())

# --- Đường cong đánh đổi ------------------------------------------------------------
print()
print('=== ĐƯỜNG CONG ĐÁNH ĐỔI ===')
print('   giá = Δ AUC tổng (âm là mất) | được = Δ AUC lớp đích')
print()
curve = delta[delta['loại'] == 'fixed μ'].sort_values('μ/β')
print(f"   {'μ':>6}  {'giá (AUC tổng)':>16}  {'được (AUC đích)':>17}  {'tỉ lệ được/giá':>16}")
for name, row in curve.iterrows():
    cost, gain = row['classifier_auc'], row['target_auc_c']
    trade = f'{gain / abs(cost):8.2f}' if abs(cost) > 1e-9 else '       -'
    print(f"   {row['μ/β']:>6.0f}  {cost:>16.3f}  {gain:>17.3f}  {trade:>16}")

print()
print(f"   {'β':>6}  {'giá (AUC tổng)':>16}  {'được (AUC đích)':>17}  {'tỉ lệ được/giá':>16}")
for name, row in delta[delta['loại'] == 'adaptive'].sort_values('μ/β').iterrows():
    cost, gain = row['classifier_auc'], row['target_auc_c']
    trade = f'{gain / abs(cost):8.2f}' if abs(cost) > 1e-9 else '       -'
    print(f"   {row['μ/β']:>6.2f}  {cost:>16.3f}  {gain:>17.3f}  {trade:>16}")

# --- Câu trả lời cho câu hỏi của người phản biện -------------------------------------
print()
print('=' * 78)
print('ADAPTIVE CÓ NẰM TRÊN ĐƯỜNG CONG KHÔNG?')
print('=' * 78)
print('Với mỗi điểm adaptive: nội suy xem ở CÙNG mức mất AUC tổng, đường cong μ cho')
print('được bao nhiêu AUC lớp đích. Dương = adaptive tốt hơn, âm = không.')
print()
fixed = curve.sort_values('classifier_auc')
for name, row in delta[delta['loại'] == 'adaptive'].iterrows():
    # np.interp cần trục x tăng dần, nên fixed đã được sort theo classifier_auc.
    reference = float(np.interp(row['classifier_auc'],
                                fixed['classifier_auc'], fixed['target_auc_c']))
    edge = row['target_auc_c'] - reference
    verdict = 'TRÊN đường cong' if edge > NOISE['target_auc_c'] else (
        'DƯỚI đường cong' if edge < -NOISE['target_auc_c'] else 'trùng đường cong (trong nhiễu)')
    print(f"  {name:<12} β={row['μ/β']:<4} : đích {row['target_auc_c']:+.3f} so với "
          f"{reference:+.3f} của đường μ  ->  {edge:+.3f}  {verdict}")

# --- Đường cong 13 lần chấm, để tham khảo -------------------------------------------
frame = pd.read_csv(METRICS_CSV)
frame = frame[frame.run != 'run']
for column in ('epoch', 'step', 'classifier_auc'):
    frame[column] = pd.to_numeric(frame[column], errors='coerce')
frame = frame.drop_duplicates(subset=['run', 'epoch', 'step'], keep='last')
print()
print('=== 13 LẦN CHẤM MỖI LẦN CHẠY (tham khảo — KHÔNG dùng để chọn checkpoint) ===')
for tag, *_ in RUNS:
    group = frame[frame.run == tag].sort_values('step')
    if not group.empty:
        print(f'  {tag:<16}', ' '.join(f'{v:6.2f}' for v in group.classifier_auc))

print()
print('=== GIÁ TRỊ TUYỆT ĐỐI ===')
print(summary[cols].round(2).to_string())
summary.to_csv(RESULT_DIR / 'adaptive_summary.csv')
print()
print('Saved:', RESULT_DIR / 'adaptive_summary.csv', '|', RESULT_DIR / 'adaptive_delta.csv')

## Ghi Chú

**Tập test được dùng đúng một lần.** Trọng số ở mục 6 sinh ra hoàn toàn từ tập train;
`--select-metric none` không nhìn test để chọn checkpoint; mục 8 chấm điểm một lượt. Đây
là điểm khác biệt về phương pháp so với vòng trước, và là phần dễ bảo vệ nhất của bài.

**`--target-classes` ở đây chỉ để gộp số khi báo cáo.** Ở chế độ `adaptive_class` nó không
còn điều khiển thứ gì model nhìn thấy — mọi lớp đều có trọng số riêng. Trainer in ra dòng
`Target classes (reporting only)` để khỏi nhầm.

**`drift` luôn bằng 0.** Đó là hiện vật của `regularizer=none`: không có anchor để so, chứ
không phải model không dịch chuyển. Đừng báo cáo cột đó.

**Cái cần nhìn trước tiên khi có kết quả.** Theo thứ tự:

1. Hệ số Spearman ở mục 6 — quyết định ablation `beta` có ý nghĩa hay không.
2. Bảng 6.1 — `beta = 0,5` có xếp ra bộ lớp khác với bốn lớp chọn tay không.
3. Bảng bội số sàn nhiễu ở mục 9 — cột nào dưới 1,0 thì bỏ, đừng diễn giải.
4. "Adaptive có nằm trên đường cong không" — đây là câu trả lời cho phản biện.

**Nếu adaptive không nằm trên đường cong.** Đó vẫn là một kết quả viết được: phương pháp
đạt mức tương đương fixed-μ mà **không cần nhìn tập test để chọn lớp**. Kết quả âm được báo
cáo trung thực vẫn tốt hơn một con số chọn lọc.

**Chi phí.** 18 lần × ~15 phút ≈ 4,5 giờ, cộng ~10 phút cho phép đo ở mục 6. Vượt một
phiên Kaggle thì dùng `ONLY_TAGS` chia nhỏ; các tag đã xong sẽ tự bỏ qua.